# 📈 informed_flow — optimize

A **daily, cross-sectional** strategy on **alternative data**. Every liquid name is ranked each
day; the book holds the top-ranked names in `max_positions` slots of one shared cash account.
Since v27 two sleeves carry the ranking — **short interest** (FINRA days-to-cover, low is
favourable) and **analyst rating changes** (Yahoo) — at fixed equal weights. Insider buying,
price-target revisions and recency are still built but ship at weight 0: on beta-adjusted forward
returns they carry no information in this universe (README, "Status (2026-09-19)"). The thesis,
the mechanics and the study history live in [README.md](./README.md).

**This notebook only optimizes.** It reads the **training window only** — the 2025+ holdout is
never loaded here — runs the Optuna study and saves its best admissible trials as *candidates*.
Whether a candidate is worth trading is decided in
[`informed_flow_diagnostics.ipynb`](./informed_flow_diagnostics.ipynb), which compares them — and
the **no-tuning control**, what `config.yaml` ships — on the deep checks, the walk-forward and
the holdout.

**Since v28 there is nothing to optimize, on purpose.** v26 searched 14 knobs for 4,000 trials
and found luck: six walk-forward refits agreed on no ranking parameter, and what they picked lost
to the market out of sample. v27 searched **one** knob (`q_out`): its train curve was flat, and
the refit picks beat the *untuned* config in only 2 of 9 out-of-sample periods. By the rule fixed
before that run, the optimizer goes: every knob is pinned, with the reason beside it in
`config.yaml`, and the "study" is one trial that records the shipped rules as the single
candidate. The cells below still work if a `search:` is ever added back — add one only with a
walk-forward that shows tuning beating not tuning.

The universe is **`Liquid500`**, point-in-time: ranked on dollar volume over 2020-H1, so every
trial is scored only from `optuna.score_from` (2020-07-01). Signals still load from 2018, so every
rolling window is warm on the first scored bar.

Flow: **load train → parameters → optimize → read the curve and the pick → boundary check → the
pick vs buy-and-hold → save candidates.**

> Data must be loaded first: run `notebooks/03_altdata_update.py`.

## 1 · Load the training panel

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from dataclasses import replace

import pandas as pd
import plotly.graph_objects as go
import yaml

from trading_simple import backtest as bt
from trading_simple import features, jobs, plots, strategies
from trading_simple import optimize as opt
from trading_simple.config import load_settings
from trading_simple.data import load_panel, symbols_with_bars
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
strat = strategies.load("informed_flow")
d = strat.data

syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]), d["bar"],
                         d["start"], d.get("end"), limit=d.get("max_symbols"))
# TRAIN ONLY, on its own window. Never load full history and slice back: a feature COLUMN
# exists only when the query window returned rows for that symbol, so a wider load gives a
# name whose only insider activity is in 2025 an all-zero column over the train years where
# this load leaves it all-NaN — holdout availability leaking into training data.
panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                   session=d.get("session", "regular"), exchanges=settings.exchanges)
features.attach_features(panel, engine, d["features"], settings=settings)

print(f"{d['universe']}: {len(syms)} symbols, {panel.close.shape[0]} daily bars "
      f"{panel.close.index[0].date()} → {panel.close.index[-1].date()}  (TRAIN, scored from "
      f"{strat.optuna.get('score_from') or d['start']})")
for name, frame in panel.features.items():
    active = int((frame.abs() > 0).any().sum())
    print(f"  feature {name:15s}: {active}/{len(syms)} symbols with data")

## 2 · Parameters and search space

Rendered straight from `config.yaml`, so it cannot drift from what runs. A parameter with a
searched range is tuned by §3; the rest ship at their value.

In [ ]:
SPACE = strat.optuna["space"]
OBJECTIVES = opt.objectives_for(strat)
HURDLE = OBJECTIVES[0] if OBJECTIVES[0] in opt.HURDLES else "excess_cagr"
print(f"study {strat.optuna['study']} · objectives {list(OBJECTIVES)} · fill model {strat.fill} · "
      f"orders of ${strat.order_notional or 10_000:,.0f} · pick floor "
      f"{strat.optuna['pick_min_trades_per_year']} trades/yr · {len(SPACE)} searched knob(s)\n")
print(f"  {'parameter':18s} {'ships':>12s}   searched")
for k, v in strat.params.items():
    spec = SPACE.get(k)
    if spec is None:
        rng = "—"
    elif spec["type"] == "categorical":
        rng = " | ".join(map(str, spec["choices"]))
    else:
        rng = f"{spec['low']} … {spec['high']}" + (" (log)" if spec.get("log") else "")
    print(f"  {k:18s} {v!s:>12s}   {rng}")

## 3 · Optimize (Optuna, resumable)

`jobs.run_optimize` runs the study in a `tsim optimize` subprocess and streams one line per
trial. It scores **`alpha_cagr`** — the return left after the beta the book actually ran
(`cagr − beta × buy-and-hold cagr`) — and nothing else. v26 also minimized `volatility`, and the
front became an exposure dial: the "balanced" pick was a book half in cash that trailed
buy-and-hold by 18%/yr on the holdout. How much of the account to invest is a sizing decision
you make afterwards, not something to search.

Every trial is scored from `optuna.score_from`, with open fills and the commission priced at the
real order size (`backtest.order_notional`). The study is journal-backed and this cell **tops up
to `optuna.n_trials` and never past it**: every extra trial is another chance to fit luck, and
the walk-forward refits in the diagnostics notebook get exactly the same budget. (v26's cell added
2,000 trials on every run; the journal reached 4,000 against a configured 1,000.)

In [ ]:
N_OPTIMIZE_TRIALS = strat.optuna["n_trials"]        # the whole budget, not a top-up
try:
    done = len(opt.load_study(strat).trials)
except KeyError:
    done = 0
if done >= N_OPTIMIZE_TRIALS:
    print(f"{strat.optuna['study']} already holds {done} trials (budget {N_OPTIMIZE_TRIALS}) — "
          "nothing to run. Bump optuna.study for a new experiment; do not top this one up.")
else:
    jobs.run_optimize("informed_flow", n_trials=N_OPTIMIZE_TRIALS - done,
                      study=strat.optuna["study"])

## 4 · Read the curve — and the pick

Few-trade parameter sets can post a flattering number by luck, so only trials trading at least
`optuna.pick_min_trades_per_year` are admissible. Among those, `opt.pick_from_front` — the one
picking rule in the repo, which the diagnostics notebook's walk-forward also uses — takes the
best, never a trial whose alpha after beta is negative, and collapses tied outcomes first. With
one objective the Pareto front is a single row, so the rule chooses among **all** admissible
trials: a best trial that trades too little then falls back to the next one instead of winning.

With one searched knob the whole study fits in one picture: alpha against the knob. **Read its
shape before its peak.** A flat curve says the knob hardly matters and the pick is a coin flip
between neighbours — then `config.yaml`'s fixed value is as good as the pick, which is exactly
what the diagnostics notebook's no-tuning control tests out of sample. A single sharp peak would
be the warning sign.

The table lists the distinct admissible outcomes, best first; a row's `trial` number is how you
name it as a candidate in the diagnostics notebook; the pick is in bold.

In [ ]:
study = opt.load_study(strat)
trials = opt.trials_frame(study)
# What the rule picks among: the Pareto front — or, with one objective, every completed trial.
front = opt.pareto_frame(study) if len(OBJECTIVES) > 1 else trials
FLOOR = strat.optuna["pick_min_trades_per_year"]
admissible = front[front["trades_per_year"] >= FLOOR]
pool = admissible if len(admissible) else front
if admissible.empty:
    print(f"⚠ NO trial trades ≥ {FLOOR}/yr — picking from everything; the diagnostics notebook "
          "FAILS this pick's completed-holds gate")
best_row = opt.pick_from_front(pool)
PICK = int(best_row["trial"])
SPACE_INT = {k for k, spec in SPACE.items() if spec["type"] == "int"}


def as_params(row) -> dict:
    """A trial row's searched parameters, cast back to the types signals() expects."""
    out = {}
    for k in SPACE:
        v = row[k]
        if SPACE[k]["type"] == "categorical":    # the declared choice itself: 42, never 42.0
            out[k] = next(c for c in SPACE[k]["choices"] if c == v)
        elif isinstance(v, str):
            out[k] = v
        elif k in SPACE_INT:
            out[k] = int(round(float(v)))
        else:
            out[k] = round(float(v), 10)          # 0.85, not Optuna's 0.8500000000000001
    return out


picked = as_params(best_row)
states = pd.Series([t.state.name for t in study.trials]).value_counts()
print(f"{len(study.trials)} trials ("
      + ", ".join(f"{n} {s.lower()}" for s, n in states.items())
      + f"); {len(admissible)} admissible (≥ {FLOOR} trades/yr), "
      f"{int(best_row['front_distinct'])} distinct outcomes")
print(f"PICK: trial {PICK} {picked}"
      + (f" — tied with {int(best_row['n_tied']) - 1} identical-scoring trial(s)"
         if best_row["n_tied"] > 1 else "")
      + ("" if best_row["beats_benchmark"] else f" — NO trial has {HURDLE} > 0"))

menu = opt.distinct_front(pool).sort_values(HURDLE, ascending=False).head(10)
cols = [c for c in ("trial", *SPACE, "alpha_cagr", "volatility", "beta", "excess_cagr",
                    "info_ratio", "max_drawdown", "trades_per_year") if c in menu]
display(menu[cols].style.hide(axis="index")
        .format({k: "{:g}" for k, spec in SPACE.items() if spec["type"] != "categorical"})
        .format({"alpha_cagr": "{:+.1%}", "excess_cagr": "{:+.1%}", "volatility": "{:.1%}",
                 "max_drawdown": "{:+.1%}", "beta": "{:.2f}", "info_ratio": "{:+.2f}",
                 "trades_per_year": "{:.0f}"})
        .apply(lambda r: ["font-weight: bold" if r["trial"] == PICK else ""] * len(r), axis=1))

if not SPACE:
    print("nothing is searched: the one trial IS config.yaml's fixed rules — nothing to pick, "
          "nothing to draw")
elif len(OBJECTIVES) == 1 and len(SPACE) == 1:
    knob = next(iter(SPACE))
    curve = trials.drop_duplicates(subset=[knob]).sort_values(knob)
    fig = go.Figure(go.Scatter(
        x=curve[knob], y=curve[HURDLE], mode="lines+markers", name=HURDLE,
        line={"color": plots.SERIES[0], "width": 2}, marker={"size": 9},
        customdata=curve[["trades_per_year", "beta", "volatility"]].to_numpy(),
        hovertemplate=(f"{knob} %{{x}}<br>{HURDLE} %{{y:+.1%}}<br>%{{customdata[0]:.0f}} trades/yr"
                       " · beta %{customdata[1]:.2f} · vol %{customdata[2]:.1%}<extra></extra>")))
    fig.add_vline(x=strat.params[knob], line={"color": "#898781", "dash": "dash", "width": 1},
                  annotation_text="config.yaml", annotation_font_color="#898781")
    fig.add_hline(y=0.0, line={"color": "#b0ada5", "width": 1})
    plots._base(fig, title=f"The whole study: train {HURDLE} against {knob} "
                           f"({curve[HURDLE].min():+.1%} … {curve[HURDLE].max():+.1%})", height=380)
    fig.update_yaxes(tickformat="+.0%", title_text=f"{HURDLE} (train)")
    fig.update_xaxes(title_text=knob)
    fig.show()
else:
    plots.plot_pareto(opt.pareto_frame(study), trials, picked=PICK).show()

### 4b · Did the search converge?

A pick is only an optimum if more trials would not have found a better one. The best-so-far
curve below should be flat for the last third of the budget; if that third still added more than
0.2 points of train alpha, the budget was too small — raise `optuna.n_trials` (and bump the
study) rather than read the pick. The diagnostics walk-forward re-runs this same search, with the
same budget, inside every period.

In [ ]:
if SPACE:
    ordered = trials.sort_values("trial")
    best = ordered[HURDLE].cummax()
    third = best.iloc[-1] - best.iloc[max(0, int(len(best) * 2 / 3) - 1)]
    found = int(ordered["trial"].iloc[int(best.to_numpy().argmax())])
    print(f"{len(best)} trials · best train {HURDLE} {best.iloc[-1]:+.2%} (trial {found}) · the "
          f"last third of the budget added {third:+.2%}")
    if strat.optuna.get("sampler") == "grid":
        print(f"✅ exhaustive grid: all {len(best)} points evaluated, the optimum is exact on it")
    else:
        print("✅ converged — the last third added < 0.2 pts" if third < 0.002 else
              "⚠ still improving — raise optuna.n_trials, bump the study and run again")
    fig = go.Figure(go.Scatter(x=ordered["trial"], y=best, mode="lines", name="best so far",
                               line={"color": plots.SERIES[0], "width": 2}))
    fig.add_trace(go.Scatter(x=ordered["trial"], y=ordered[HURDLE], mode="markers",
                             name="each trial", marker={"size": 5, "color": plots.SERIES[2]}))
    plots._base(fig, title=f"Convergence: best train {HURDLE} so far", height=320)
    fig.update_yaxes(tickformat="+.1%")
    fig.show()
else:
    print("nothing is searched — nothing to converge")

## 5 · Boundary check

If the pick — or at least half of the distinct outcomes — sits on the edge of a searched range,
the real optimum may lie outside it. **For `q_out` that is not an instruction to widen.** Above
0.85 the hysteresis band between `q_in` (0.90) and `q_out` is gone and the book churns on rank
noise; below 0.60 it is holding names from the middle of the ranking, where there is nothing to
earn (only the top fifth pays). A pick on either edge is a statement about how cheap or dear
turnover is under the cost model — check it against §3a's real-order-size column and §3d's
stability table in the diagnostics notebook, and prefer `config.yaml`'s fixed value if the curve
in §4 is flat.

In [ ]:
if SPACE:
    report = opt.boundary_report(picked, SPACE, front=pool)
    display(report)
    flagged = report[report["action"] != "ok"]
    print("no bound is pressing — the search box holds the optimum" if flagged.empty else
          f"on an edge: {dict(flagged['action'])} — read the note above before widening anything")
else:
    print("no searched knob, so no bound to check")

## 6 · The pick on training data — against buy-and-hold

A sanity look, not a verdict: this is the window the study was fitted on (scored from
`optuna.score_from`, like every trial), so every number here
flatters the pick by construction. The deep checks and the untouched holdout are in the
diagnostics notebook. Costs are on, and the dashed line is the mandatory baseline — equal-weight
buy-and-hold of the same names (`strategies/baseline`), paying the same costs.

In [ ]:
INIT_CASH = settings.backtest.init_cash
SCORE_FROM = strat.optuna.get("score_from") or d["start"]
res = bt.run_strategy(strat, panel, params=picked, settings=settings, score_from=SCORE_FROM)
# The baseline under THIS strategy's execution rules (fill model, order size), so the table
# compares like with like — it is then the engine's own benchmark, number for number.
res_base = bt.run_strategy(replace(strategies.load("baseline"), fill=strat.fill,
                                   order_notional=strat.order_notional),
                           panel, settings=settings, score_from=SCORE_FROM)
st = bt.evaluate(strat, panel, picked, settings=settings, score_from=SCORE_FROM)
base = bt.portfolio_stats(res_base)
rows = ["cagr", "volatility", "max_drawdown", "return / vol", "sharpe", "trades_per_year"]
for stats in (st, base):
    stats["return / vol"] = stats["cagr"] / stats["volatility"]
summary = pd.DataFrame({f"pick #{PICK}": [st[k] for k in rows],
                        "buy & hold": [base[k] for k in rows]}, index=rows)
display(summary.style.format("{:+.1%}", subset=pd.IndexSlice[rows[:3], :])
        .format("{:.2f}", subset=pd.IndexSlice[rows[3:], :]))
print(f"alpha after beta {st['alpha_cagr']:+.2%}/yr (beta {st['beta']:.2f}) · excess over "
      f"buy-and-hold {st['excess_cagr']:+.2%}/yr · information ratio {st['info_ratio']:+.2f} · "
      f"{int(st['n_trades'])} trades")

fig = go.Figure()
fig.add_trace(plots._scatter(
    bt.equity_curve(res_base, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0,
    name="baseline (buy & hold)", line={"color": "#898781", "dash": "dash", "width": 1.5},
    hovertemplate="%{x|%Y-%m-%d}<br>%{y:+.1%}<extra>baseline</extra>"))
fig.add_trace(plots._scatter(
    bt.equity_curve(res, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0,
    name=f"pick #{PICK}", line={"color": plots.SERIES[0], "width": 1.5},
    hovertemplate="%{x|%Y-%m-%d}<br>%{y:+.1%}<extra>pick</extra>"))
fig.update_xaxes(type="date")  # _scatter packs x as epoch ms; this is what keeps it a date axis
plots._base(fig, title=(f"{d['universe']} · train from {SCORE_FROM} — "
                        "the pick vs equal-weight buy-and-hold"),
            height=420)
fig.update_yaxes(tickformat="+.0%", title_text="cumulative return")
fig.show()
display(bt.symbol_stats(res).head(10).round(3))

## 7 · Save the candidates for the diagnostics notebook

The best distinct admissible trials (up to ten) are written to `studies/<study>.candidates.yaml`
with their parameters, their training numbers, which one the picking rule chose, the scored window
and whether the pick was admissible (a study with no trial above the trade floor falls back past
it). The file is a snapshot; it is git-ignored, like the journal it came from.

**Next:** open `informed_flow_diagnostics.ipynb` and Run All. Its default `CANDIDATES = ["pick",
"config"]` puts the pick beside what `config.yaml` ships — the no-tuning control.

In [ ]:
path = opt.study_path(strat, strat.optuna["study"]).with_suffix(".candidates.yaml")
METRICS = ("alpha_cagr", "volatility", "beta", "excess_cagr", "info_ratio", "tracking_error",
           "cagr", "max_drawdown", "trades_per_year", "years")
snapshot = {
    "study": strat.optuna["study"],
    "saved": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
    "pick_min_trades_per_year": FLOOR,
    "score_from": strat.optuna.get("score_from") or d["start"],
    "admissible": bool(len(admissible)) and bool(best_row["beats_benchmark"]),
    "pick": PICK,
    "front": [{"trial": int(r["trial"]),
               **{m: round(float(r[m]), 6) for m in METRICS if m in r},
               "params": as_params(r)}
              for _, r in menu.iterrows()],
}
path.write_text(yaml.safe_dump(snapshot, sort_keys=False))
print(f"saved {len(snapshot['front'])} candidates (pick #{PICK}) -> {path}")
print("trials:", [c["trial"] for c in snapshot["front"]])